In [2]:
import pandas as pd
import mysql.connector
from pathlib import Path

file_list = [
    "C:/skn29/PYTHON/project/data/eco_car_table_with_attribute.xlsx",
    "C:/skn29/PYTHON/project/data/imported_car_table_with_attribute.xlsx",
    "C:/skn29/PYTHON/project/data/korean_car_table_with_attribute.xlsx"
]

# 2) 파일 읽기
df_list = [pd.read_excel(file) for file in file_list]
df = pd.concat(df_list, ignore_index=True)

# 3) 컬럼명 변경
df = df.rename(columns={
    'vehicle_type': 'body_type',
    'maker': 'maker_name'
})

# 4) 필요한 컬럼만 선택
df = df[
    [
        'maker_name',
        'model_name',
        'origin_type',
        'body_type',
        'vehicle_class',
        'base_price',
        'eco_flag',
        'eco_fuel_type'
    ]
].copy()

# 5) 문자열 컬럼 공백 제거
str_cols = [
    'maker_name', 'model_name', 'origin_type',
    'body_type', 'vehicle_class', 'eco_flag', 'eco_fuel_type'
]

for col in str_cols:
    df[col] = df[col].astype(str).str.strip()

# 6) eco_fuel_type 처리: "NULL" 문자열을 실제 NULL로 변경
df['eco_fuel_type'] = df['eco_fuel_type'].replace({
    'NULL': None,
    'null': None,
    'None': None,
    'nan': None,
    '': None
})

# 7) eco_flag 보정
df['eco_flag'] = df['eco_fuel_type'].apply(
    lambda x: 'Y' if x in ['전기', '하이브리드'] else 'N'
)

# 8) 숫자형 변환
df['base_price'] = pd.to_numeric(df['base_price'], errors='coerce')

# 9) 필수값 없는 행 제거
df = df.dropna(subset=[
    'maker_name', 'model_name', 'origin_type',
    'body_type', 'vehicle_class', 'base_price'
])

# 10) 정수형 변환
df['base_price'] = df['base_price'].astype(int)

# 11) 중복 제거
df = df.drop_duplicates(subset=['maker_name', 'model_name'])

# 12) 확인
print("적재 대상 행 수:", len(df))
print(df.head())
print(df['eco_fuel_type'].value_counts(dropna=False))

# 13) MySQL 연결
conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password='!didwjdgus16',
    database='car_insurance_final',
    charset='utf8mb4'
)
cursor = conn.cursor()

# 14) INSERT SQL
insert_sql = """
INSERT INTO car_master (
    maker_name,
    model_name,
    origin_type,
    body_type,
    vehicle_class,
    base_price,
    eco_flag,
    eco_fuel_type
) VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
    origin_type = VALUES(origin_type),
    body_type = VALUES(body_type),
    vehicle_class = VALUES(vehicle_class),
    base_price = VALUES(base_price),
    eco_flag = VALUES(eco_flag),
    eco_fuel_type = VALUES(eco_fuel_type),
    updated_at = CURRENT_TIMESTAMP
"""

# 15) executemany용 데이터 생성
data_to_insert = [
    (
        row.maker_name,
        row.model_name,
        row.origin_type,
        row.body_type,
        row.vehicle_class,
        row.base_price,
        row.eco_flag,
        row.eco_fuel_type
    )
    for row in df.itertuples(index=False)
]

# 16) INSERT 실행
cursor.executemany(insert_sql, data_to_insert)
conn.commit()

print(f"{cursor.rowcount}건 처리 완료")

# 17) 결과 확인
cursor.execute("SELECT COUNT(*) FROM car_master")
print("car_master 전체 건수:", cursor.fetchone()[0])

cursor.execute("""
SELECT maker_name, model_name, origin_type, body_type, vehicle_class, base_price, eco_flag, eco_fuel_type
FROM car_master
ORDER BY car_id DESC
LIMIT 10
""")

for row in cursor.fetchall():
    print(row)

# 18) 종료
cursor.close()
conn.close()

적재 대상 행 수: 4086
  maker_name   model_name origin_type body_type vehicle_class  base_price  \
0         기아          EV3          국산        승용            소형       42527   
1         기아          EV6          국산        승용            중형       50268   
2         기아          EV9          국산        승용            대형       66104   
3         기아  K5플러그인하이브리드          국산        승용            중형       31775   
4         기아      K5하이브리드          국산        승용            중형       28069   

  eco_flag eco_fuel_type  
0        Y            전기  
1        Y            전기  
2        Y            전기  
3        Y         하이브리드  
4        Y         하이브리드  
eco_fuel_type
None     3604
하이브리드     287
전기        174
CNG        15
수소          6
Name: count, dtype: int64
4106건 처리 완료
car_master 전체 건수: 4066
('힐링코리아', '아드리아(ADRIA)캠핑트레일러', '국산', '승합', '다인승', 33580, 'N', None)
('홍카맨캠핑카', '홍카맨캠핑카', '국산', '승합', '다인승', 38250, 'N', None)
('홍성자동차', '홍성캠핑트레일러', '국산', '승합', '다인승', 19956, 'N', None)
('홍성기업', '홍성캠핑트레일러', '국산', '승